# Public notebook variant
Dataset records, captions, image hashes and Gemma references are loaded from a private input file rather than embedded. Prepare gemma560_private_inputs.json locally using prepare_colab_inputs.py and upload it privately to Colab. Set GEMMA_560_INPUTS if using another path. Original research notebook remains in the complete handoff package.


# Hoàn tất so sánh chính Gemma vs SigLIP2-224 trên 560 ảnh

Notebook này **chưa được chạy**. Chọn GPU trong Colab rồi Run all. Tải ảnh COCO và model pinned revision; cần mạng cho Hugging Face và COCO. Dữ liệu đầu vào và điểm Gemma đã được nhúng, không phụ thuộc file của phiên ChatGPT.

Gemma được giữ nguyên từ vector float32 đã kiểm chứng. Chỉ SigLIP2-224 được inference mới; không dùng SigLIP2-256 thay thế. 560 ảnh, caption, thứ tự và hash đều được cố định. Notebook xuất McNemar exact và paired bootstrap 10.000 lần của so sánh chính theo công thức CLIPScore đã dùng ở stage 4. Không có kết quả inference hay timing mới cho tới khi chạy thành công.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.19.0', 'huggingface_hub==1.33.0', 'numpy', 'scipy', 'pillow', 'safetensors', 'sentencepiece', 'accelerate'])


In [ ]:
import base64, hashlib, json
from pathlib import Path
root=Path('/content/embeddinggemma2_560') if Path('/content').exists() else Path.cwd()/'embeddinggemma2_560'
root.mkdir(parents=True, exist_ok=True)
import os
input_file = Path(os.environ.get("GEMMA_560_INPUTS", "/content/gemma560_private_inputs.json"))
if not input_file.is_file():
    raise FileNotFoundError("Supply private input generated with prepare_colab_inputs.py; see README.")
payload = json.loads(input_file.read_text())
for name,item in payload.items():
    data=base64.b64decode(item['b64'])
    assert hashlib.sha256(data).hexdigest()==item['sha256']
    (root/name).write_bytes(data)
print('Restored fixed 560 manifest and verified Gemma reference:', root)


In [ ]:
import torch
print('Torch', torch.__version__, 'GPU available:', torch.cuda.is_available())
subprocess.check_call([sys.executable, str(root/'run_siglip224.py')])


In [ ]:
import zipfile
result=json.loads((root/'primary_siglip224_results.json').read_text())
print(json.dumps(result, indent=2))
output=root.parent/'siglip224_confirmation_results.zip'
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['primary_siglip224_results.json','frozen_manifest.json','frozen_protocol.json','primary_reference.json','run_siglip224.py','runs/siglip224/embeddings.npz']:
        archive.write(root/name, name)
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    print(output)
